In [3]:
import operator
from typing import Annotated, List, TypedDict, Union
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
import os

from agent_data import BatteryDataAgent
from agent_knowledge import BatteryKnowledgeAgent

# 1. 상태(State) 정의: 에이전트들이 공유할 데이터 가방
class AgentState(TypedDict):
    query: str                   # 사용자 질문
    db_results: str              # Data Agent가 찾은 수치 데이터
    manual_results: str          # Knowledge Agent가 찾은 가이드북 지식
    next_step: str               # 다음에 어디로 갈지 (router가 결정)
    final_answer: str            # 사용자에게 줄 최종 답변

# 2. 팀장(Orchestrator) 노드 정의
class BatteryOrchestrator:
    def __init__(self):
        self.llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
        # 앞서 만든 에이전트 인스턴스 (실제 환경에선 import해서 사용)
        self.data_agent = BatteryDataAgent(
            db_path = "/home/jovyan/project/data/battery_factory.db",
            dict_path = "/home/jovyan/project/data/battery_semantic_dict.json"
        ) 
        self.knowledge_agent = BatteryKnowledgeAgent()

    def router(self, state: AgentState):
        """질문의 성격을 파악하여 전문가를 배정합니다."""
        print("🤖 팀장: 질문 분석 중...")
        prompt = f"""
        사용자의 질문을 보고 어떤 전문가가 필요한지 결정하세요.
        1. 수치 데이터 조회나 통계가 필요하면: 'DATA'
        2. 불량 원인, 조치 방법, 매뉴얼 지식이 필요하면: 'KNOWLEDGE'
        3. 둘 다 필요하면: 'BOTH'
        
        질문: {state['query']}
        결과(DATA/KNOWLEDGE/BOTH만 출력):"""
        
        decision = self.llm.invoke(prompt).content.strip().upper()
        return {"next_step": decision}

    def call_data_expert(self, state: AgentState):
        """숫자 요원 호출"""
        print("📊 숫자 요원: DB에서 수치 분석 중...")
        res = self.data_agent.execute_and_analyze(state['query'])
        return {"db_results": res}

    def call_knowledge_expert(self, state: AgentState):
        """지식 요원 호출"""
        print("📚 지식 요원: 가이드북에서 원인 검색 중...")
        res = self.knowledge_agent.generate_answer(state['query'])
        return {"manual_results": res}

    def final_responder(self, state: AgentState):
        """모든 결과를 종합하여 최종 답변 작성"""
        print("🏁 팀장: 최종 보고서 작성 중...")
        summary_prompt = f"""
        당신은 통합 관제 시스템입니다. 아래 정보를 종합하여 사용자에게 최종 답변을 하세요.
        
        [수치 데이터 분석]: {state.get('db_results', '없음')}
        [가이드북 지식]: {state.get('manual_results', '없음')}
        
        조회된 결과를 바탕으로 현상을 진단하고 권장 조치를 포함하여 답변하세요.
        """
        response = self.llm.invoke(summary_prompt)
        return {"final_answer": response.content}

# 3. LangGraph 워크플로우 구성
orchestrator = BatteryOrchestrator()
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("router", orchestrator.router)
workflow.add_node("data_expert", orchestrator.call_data_expert)
workflow.add_node("knowledge_expert", orchestrator.call_knowledge_expert)
workflow.add_node("final_responder", orchestrator.final_responder)

# 엣지 연결 (로직 설계)
workflow.set_entry_point("router")

# 조건부 엣지: router의 결정에 따라 경로 분기
def route_decision(state):
    if state["next_step"] == "DATA": return "data_expert"
    if state["next_step"] == "KNOWLEDGE": return "knowledge_expert"
    return "data_expert" # BOTH 혹은 기본값

workflow.add_conditional_edges(
    "router",
    route_decision,
    {
        "data_expert": "data_expert",
        "knowledge_expert": "knowledge_expert"
    }
)

# 전문가 작업 후 최종 응답으로 연결
workflow.add_edge("data_expert", "final_responder")
workflow.add_edge("knowledge_expert", "final_responder")
workflow.add_edge("final_responder", END)

# 4. 컴파일 및 실행
app = workflow.compile()

# 실행 예시
input_data = {"query": "1000번 팩의 전압이 현재 3.2V인데, 이게 왜 문제고 어떻게 고쳐야 해?"}
for output in app.stream(input_data):
    print(output)

🤖 팀장: 질문 분석 중...
{'router': {'next_step': 'BOTH'}}
📊 숫자 요원: DB에서 수치 분석 중...
📡 생성된 SQL: 
SELECT 
    M01CV01, M01CV02, M01CV03, M01CV04, M01CV05, M01CV06, M01CV07, M01CV08, M01CV09, M01CV10
FROM 
    battery_logs
WHERE 
    SerialNumber = 1000
ORDER BY 
    Time DESC
LIMIT 1;

{'data_expert': {'db_results': '1000번 팩의 전압이 현재 3.2V로 측정된 것으로 보아, 이는 정상적인 전압 범위에서 벗어난 값입니다. 일반적으로, 리튬 이온 배터리 팩의 전압은 3.7V에서 4.2V 사이로 유지되는 것이 이상적입니다. 3.2V는 이 범위에서 낮은 값으로, 배터리 셀의 상태나 충전/방전 과정에서 문제가 있을 수 있습니다.\n\n위의 데이터를 살펴보면, M01CV01부터 M01CV10까지의 전압 값은 모두 3.53V 근처로 집중되어 있습니다. 이는 대부분의 셀들이 정상적인 전압을 유지하고 있음을 나타냅니다. 그러나 1000번 팩의 전압이 3.2V로 측정된 경우, 이는 해당 셀에 문제가 있거나, 충전/방전 과정에서 이상이 발생했을 수 있습니다.\n\n이 문제를 해결하기 위해서는 다음의 단계를 고려할 수 있습니다:\n\n1. **셀의 상태 확인**: 1000번 팩의 셀을 개별적으로 확인하여 내부 저항, 용량, 자기 방전 특성을 측정합니다. 이는 셀의 상태를 평가하고 문제의 원인을 파악하는 데 도움이 됩니다.\n\n2. **충전/방전 과정 확인**: 충전과 방전의 과정에서 문제가 발생했을 수 있습니다. 충전/방전 회로와 알고리즘을 확인하여 이상이 없는지 검토합니다.\n\n3. **셀 밸런싱**: 배터리 팩의 셀들은 서로 전압을 균일하게 유지해야 합니다. 셀 밸런싱을 통해 전압 차이를 조정할 수 있습니다. 이는 특정 셀이 다른 셀보다 더 